<a href="https://colab.research.google.com/github/yhou151209/541_project/blob/main/541.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ortools

In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Small demo dataset for Phase 1.
    You can replace this with JSON-loaded data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "max_shifts": 2,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            # Monday
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},

            # Tuesday
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        key = (row["employee_id"], row["day"], row["shift"])
        lookup[key] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Build a schedule from scratch with OR-Tools CP-SAT.

    Hard constraints:
    1. Required coverage must be met exactly.
    2. Employee must be available for assigned shift.
    3. Employee must have the required skill/role.
    4. Employee cannot hold two roles in the same day/shift.
    5. Employee cannot exceed max_shifts.

    Soft objective:
    - Minimize the maximum number of assigned shifts to keep load more balanced.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)

    model = cp_model.CpModel()

    # Decision variable:
    # x[(emp_id, day, shift, role)] = 1 if employee is assigned to that role in that shift.
    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            emp_skills = set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)
            has_skill = role in emp_skills

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            # Not available or not qualified -> cannot assign
            if (not is_available) or (not has_skill):
                model.Add(var == 0)

    # Coverage constraints: exactly meet required_count
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        eligible_vars = [
            x[(emp["id"], day, shift, role)]
            for emp in staff
        ]
        model.Add(sum(eligible_vars) == required_count)

    # Same employee cannot take two roles in the same shift
    unique_shift_keys = sorted({(r["day"], r["shift"]) for r in requirements})
    roles = sorted({r["role"] for r in requirements})

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in unique_shift_keys:
            vars_same_shift = [
                x[(emp_id, day, shift, role)]
                for role in roles
                if (emp_id, day, shift, role) in x
            ]
            if vars_same_shift:
                model.Add(sum(vars_same_shift) <= 1)

    # Max shifts per employee
    for emp in staff:
        emp_id = emp["id"]
        max_shifts = int(emp["max_shifts"])
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(sum(emp_vars) <= max_shifts)

    # Soft fairness objective:
    # minimize the maximum assigned shifts among employees
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    for emp in staff:
        emp_id = emp["id"]
        total = model.NewIntVar(0, len(unique_shift_keys), f"total_{emp_id}")
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(total == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total

    max_load = model.NewIntVar(0, len(unique_shift_keys), "max_load")
    for emp_id, total in total_assigned_per_emp.items():
        model.Add(total <= max_load)

    model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found with current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def print_schedule(schedule: List[Dict[str, str]]) -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print("\n===== SCHEDULE =====")
    for day_shift in sorted(grouped.keys()):
        day, shift = day_shift
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[day_shift], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("====================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> List[Dict[str, str]]:
    """
    Simple Phase 1 update:
    supports requests like:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Strategy:
    1. Mark that employee unavailable for the target shift.
    2. Re-run schedule generation from scratch.
       This is the simplest robust Phase 1 approach.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"]
    day = change["day"]
    shift = change["shift"]

    # Find employee id
    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    # Copy data deeply enough for this use case
    new_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found_availability_row = False
    for row in new_data["availability"]:
        if row["employee_id"] == emp_id and row["day"] == day and row["shift"] == shift:
            row["available"] = False
            found_availability_row = True

    if not found_availability_row:
        new_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    # Regenerate whole schedule for simplicity
    updated_schedule = generate_schedule(new_data)
    return updated_schedule


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule)

    # Manual Phase 1 test: Amy unavailable Monday evening
    change_request = {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening",
    }

    print("Applying update: Amy unavailable Monday evening...")
    updated = update_schedule(data, schedule, change_request)
    print_schedule(updated)


if __name__ == "__main__":
    main()

Generating initial schedule...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> David (E05)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> David (E05)

Applying update: Amy unavailable Monday evening...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Amy (E01)

Tuesday - evening
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> David (E05)
  server   -> Cathy (E04)



In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Demo restaurant dataset for Phase 1.
    Replace this with your real restaurant data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E06",
                "name": "Emma",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 2,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": False},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E06", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E06", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    """
    Convert schedule list into a set of:
    (employee_id, day, shift, role)
    """
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    General solver.

    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Exact coverage
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # One employee cannot do two roles in same shift
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Max shifts
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Fairness: minimize max workload
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for emp_id, total_var in total_assigned_per_emp.items():
        model.Add(total_var <= max_load)

    # Keep-as-many-existing-assignments-as-possible objective
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    # Lexicographic-style weighted objective
    # Strongly prefer keeping old assignments, secondarily reduce max load
    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """
    Currently supports:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Update strategy:
    1. Modify availability.
    2. Re-solve, but use existing_schedule as a preference,
       so the solver tries to keep as many old assignments as possible.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if row["employee_id"] == emp_id and row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower():
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    """
    Command-line input for Phase 1 testing.
    """
    print("Enter an update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule, title="INITIAL SCHEDULE")

    answer = input("Do you want to enter an unavailable update? (y/n): ").strip().lower()
    if answer != "y":
        print("Done.")
        return

    try:
        change_request = prompt_unavailable_change()
        updated_schedule, updated_data = update_schedule(data, schedule, change_request)

        print_schedule(updated_schedule, title="UPDATED SCHEDULE")
        compare_schedules(schedule, updated_schedule)

        # Optional: update in-memory state so more edits can be chained later
        data = updated_data
        schedule = updated_schedule

    except ValueError as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    main()

Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

Do you want to enter an unavailable update? (y/n): y
Enter an update request.
Employee name: Emma
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): morning

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> David (E05)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  - E06: Tuesday morning cashier
Added assignments:
  + E05: Tuesday morning cashier



In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Demo restaurant dataset for Phase 1.
    Replace this with your real restaurant data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E06",
                "name": "Emma",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 2,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": False},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E06", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E06", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    General solver.

    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for emp_id, total_var in total_assigned_per_emp.items():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """
    Currently supports:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Update strategy:
    1. Modify availability.
    2. Re-solve, but use existing_schedule as a preference,
       so the solver tries to keep as many old assignments as possible.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if (
            row["employee_id"] == emp_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    print("\nEnter an unavailable update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule, title="INITIAL SCHEDULE")

    while True:
        should_update = ask_yes_no("Do you want to enter an unavailable update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_unavailable_change()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            # Keep latest state for the next round
            data = updated_data
            schedule = updated_schedule

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")


if __name__ == "__main__":
    main()

Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

Do you want to enter an unavailable update? (y/n): y

Enter an unavailable update request.
Employee name: Betty
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): evening

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  - E03: Tuesday evening cashier
Added assignments:
  + E06: Tuesday evening cashier

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if (
            row["employee_id"] == emp_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    print("\nEnter an unavailable update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")
    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")

    while True:
        should_update = ask_yes_no("Do you want to enter an unavailable update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_unavailable_change()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updated availability back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> David (E05)

Monday - morning
  cashier  -> Emma (E06)
  server   -> John (E02)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Do you want to enter an unavailable update? (y/n): y

Enter an unavailable update request.
Employee name: Cathy
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): morning

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> David (E05)

Monday - morning
  cashier  -> Emma (E06)
  server   -> John (E02)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  